# 01 — Raw IEEE-CIS data understanding

Our original exploration had four goals: (1) identify the transaction grain and target, (2) group fields by **semantic role**, (3) quantify class imbalance and identity coverage, and (4) propose sensible *relationship keys* rather than automatically connecting equal values.

All fraud-rate profiles in this notebook are **exploratory only**, never fed directly into a model. The original exploration included the entire Kaggle training table; see `docs/methodology.md` for subsequent-hypothesis caveats.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Repository:", ROOT)

In [ ]:
import pandas as pd
from fraud_graph.paths import RAW_DIR

transactions = pd.read_csv(ROOT / "data/raw/train_transaction.csv")
identity = pd.read_csv(ROOT / "data/raw/train_identity.csv")
print("Transaction table:", transactions.shape)
print("Identity table:", identity.shape)
print("Fraud prevalence:\n", transactions.isFraud.value_counts(normalize=True))
print("Identity coverage:", transactions.TransactionID.isin(identity.TransactionID).mean())

## Semantics and causal safety

- `TransactionID` uniquely names rows, not a continuous predictive quantity.
- `TransactionDT` is a **relative time offset**, used for chronological splitting and directed graph edges.
- `card1`...`card6`, `addr1`/`addr2`, `DeviceInfo` need semantic treatment; a float dtype does not prove a quantity is continuous.
- High-cardinality keys can define relationships but may form enormous cliques; estimate group sizes and potential clique edges *before* graph construction.
- Full-period group fraud rates must never become direct features: they contain future labels.

In [ ]:
assert transactions.TransactionID.is_unique
assert identity.TransactionID.is_unique
print("Fraud counts:", transactions.isFraud.value_counts().to_dict())
print("Time range:", transactions.TransactionDT.min(), transactions.TransactionDT.max())
print("Missing identity rows:", len(transactions) - transactions.TransactionID.isin(identity.TransactionID).sum())

In [ ]:
from IPython.display import display

def profile_relation(df, column, min_group_size=100):
    counts = df[column].dropna().value_counts()
    potential_clique_edges = int((counts * (counts - 1) // 2).sum())
    profiles = (df.dropna(subset=[column]).groupby(column).agg(
        transactions=("TransactionID", "count"),
        frauds=("isFraud", "sum"),
        fraud_rate=("isFraud", "mean"),
        first_time=("TransactionDT", "min"),
        last_time=("TransactionDT", "max"),
    ))
    profiles["span_days"] = (profiles.last_time - profiles.first_time) / 86400
    large = profiles[profiles.transactions >= min_group_size]
    print(f"{column}: coverage={df[column].notna().mean():.3f} "
          f"unique={len(counts):,} clique_edges={potential_clique_edges:,}")
    display(counts.describe(percentiles=[.5, .75, .9, .95, .99]))
    display(large.fraud_rate.describe(percentiles=[.5, .75, .9, .95, .99]))
    return profiles

for col in ["card1", "card2", "card3", "card4", "card5", "card6", "addr1", "addr2"]:
    _ = profile_relation(transactions, col)

## Composite keys

We settled on `(card1, addr1)` (broad, strong repeat coverage) and `(card1, DeviceInfo)` (specific but sparse) as two *inferred relations*. A shared `DeviceInfo` does **not** guarantee shared physical device identity. Neither relation uses target labels to create edges.

In [ ]:
df = transactions[["TransactionID", "TransactionDT", "isFraud", "card1", "addr1"]].merge(
    identity[["TransactionID", "DeviceInfo"]], on="TransactionID", how="left", validate="one_to_one"
)
for columns in [("card1", "addr1"), ("card1", "DeviceInfo")]:
    counts = df.dropna(subset=list(columns)).groupby(list(columns)).size()
    print(columns,
          "coverage=", f"{df[list(columns)].notna().all(axis=1).mean():.2%}",
          "unique keys=", len(counts),
          "repeated keys=", int((counts >= 2).sum()),
          "transactions in repeated keys=", int(counts[counts >= 2].sum()))